In [19]:
#!/usr/bin/env python

import ctypes as cts
import sys

import numpy as np
import time

from scipy.spatial.distance import directed_hausdorff



def main(*argv):

    rows0 = 3000
    cols0 = 3
    kernel_size = 3
    rows1 = 1200
    cols1 = 3
    mat0 = np.random.randn(rows0, cols0).astype(cts.c_double)
    mat1 = np.random.randn(rows1, cols1).astype(cts.c_double)
    
    grid = generate_meshgrid(mat0,mat1,4,4,0.1)
    #x = jaccard_similarity1(mat0,mat1,1,1,1)
    #print(mat1)
    #print(grid)
    #grid1 = grid.astype(cts.c_double)
    grid1 = np.ascontiguousarray(grid)
    #print(grid1)
    r1,c1 = grid1.shape
    mat_res,dealloc_array = binary(mat0, rows0, cols0, mat1, rows1, cols1)
    
    start = time.time()
    mat_res,dealloc_array = binary(mat0, rows0, cols0, grid1, r1, c1)
    end = time.time()
    
    print(mat_res.flatten().astype(bool))
    print(f"Time: {end - start}")
    """
    start = time.time()
    mat_res,dealloc_array = haus(mat0, rows0, cols0, mat1, rows1, cols1)
    print("Johnny")
    print(max(mat_res))
    end = time.time()
    print(f"Time: {end - start}")
    
    dealloc_array(mat_res)
    """
    print("\n")
    print("Scipy")
    start = time.time()
    scipy__dist = directed_hausdorff(mat0, mat1)
    print(scipy__dist)
    end = time.time()
    
    print(f"Time: {end - start}")
    start = time.time()
    #yasen = hausdorff_distance(mat0, mat1)
    print("\n")
    print("Yasen")
    #print(max(yasen))
    end = time.time()
    print(f"Yasen Time: {end - start}")
    
if __name__ == "__main__":
    print("Python {:s} {:03d}bit on {:s}\n".format(" ".join(elem.strip() for elem in sys.version.split("\n")),
                                                   64 if sys.maxsize > 0x100000000 else 32, sys.platform))
    rc = main(*sys.argv[1:])
    #sys.exit(rc)

Python 3.8.5 (default, Sep  3 2020, 21:29:08) [MSC v.1916 64 bit (AMD64)] 064bit on win32

Size of grid: (260, 3)
<class 'ctypes.CDLL'>
<class 'ctypes.CDLL'>
[False False False False False False False  True False False False False
 False  True False False False  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True False  True  True  True
  True False  True False False False False False False False False False
 False False  True False False  True False False False False False False
 False False False  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True False False False False  True False  True False False
 False False False False False False Fa

In [18]:

DLL_NAME = "./haus_46.{:s}".format("dll" if sys.platform[:3].lower() == "win" else "so")

def np_mat_type(rows, cols, element_type=float):
    return np.ctypeslib.ndpointer(dtype=element_type, shape=(rows, cols), flags="C_CONTIGUOUS")


def haus(mat0, rows0, cols0, mat1, rows1, cols1):

    #mat0 = np.array([[1,2,3],[2,2,2],[3,3,3],[1,4,3],[5,2,3]]).astype(cts.c_double)
    #mat1 = np.array([[4,4,4],[5,4,4],[6,4,4],[7,4,4],[8,4,4],[9,4,4]]).astype(cts.c_double)
    
    
    dll = cts.CDLL(DLL_NAME)
    matrix_func = dll.pointwiseDistance
    matrix_func.argtypes = (
        np_mat_type(rows0, cols0), cts.c_size_t, cts.c_size_t,
        np_mat_type(rows1, cols1), cts.c_size_t, cts.c_size_t)
    
    #all point
    matrix_func.restype = np_mat_type(rows0, 1)
    #one point
    #matrix_func.restype = np_mat_type(1, 1)
    
    
    dealloc_array = dll.deallocArray
    #all point
    dealloc_array.argtypes = (np_mat_type(rows0, 1),)
    #one point
    #dealloc_array.argtypes = (np_mat_type(1, 1),)
    
    dealloc_array.restype = None

    #print("mat0:")
    #print(mat0)
    #print("\nmat1:")
    #print(mat1)
    start = time.time()
    mat_res = matrix_func(mat0, rows0, cols0, mat1, rows1, cols1)
    return mat_res,dealloc_array
    #dealloc_array(mat_res)


    
def generate_meshgrid(P, Q, res_x,res_y,res_z):
    """
    A function used to generate a meshgrid over two arrays given a resolution
    ...

    Arguments
    -------
    P: np.array
        An array where each entry stores an x,y,z coordination of the point cloud
    Q: np.array
        An array where each entry stores an x,y,z coordination of the point cloud
    res_x: float
        given the resolution/stepsize in x direction
    res_y: float
        given the resolution/stepsize in y direction
    res_z: float
        given the resolution/stepsize in z direction

    Return: np.array/int
        returns an array of the grid mesh points
    """
        

    #get boundaries of grid
    x_min = min(min(P[:, 0]), min(Q[:, 0])); x_max = max(max(P[:, 0]), max(Q[:, 0]))
    y_min = min(min(P[:, 1]), min(Q[:, 1])); y_max = max(max(P[:, 1]), max(Q[:, 1]))
    z_min = min(min(P[:, 2]), min(Q[:, 2])); z_max = max(max(P[:, 2]), max(Q[:, 2]))
    
    #apply grid resolution
    xx = np.arange(x_min+.25, x_max-.25, res_x) 
    yy = np.arange(y_min+.25, y_max-.25, res_y)
    zz = np.arange(z_min+.25, z_max-.25, res_z)

    #generate grid
    xx, yy, zz = np.meshgrid(xx,yy,zz)
    grid = np.array([xx.ravel(),yy.ravel(),zz.ravel()]).T

    print(f"Size of grid: {grid.shape}")
    return grid



def binary(mat0, rows0, cols0, grid, rowsg, colsg):

    dll = cts.CDLL(DLL_NAME)
    print(type(dll))
    matrix_func = dll.binaryMaskGenerator
    matrix_func.argtypes = (
        np_mat_type(rows0, cols0), cts.c_size_t, cts.c_size_t,
        np_mat_type(rowsg, colsg), cts.c_size_t, cts.c_size_t)
    
    matrix_func.restype = np_mat_type(rowsg, 1,element_type=int)

    dealloc_array = dll.deallocArray
    #all point
    dealloc_array.argtypes = (np_mat_type(rowsg, 1,element_type=int),)

    dealloc_array.restype = None

    start = time.time()
    mat_res = matrix_func(mat0, rows0, cols0, grid, rowsg, colsg)
    
    return mat_res,dealloc_array
    #dealloc_array(mat_res)

In [5]:
from scipy.spatial.distance import directed_hausdorff
import numpy as np


rows0 = 12000
cols0 = 3
kernel_size = 3
rows1 = 30000
cols1 = 3

mat0 = np.random.randn(rows0, cols0)

mat1 = np.random.randn(rows1, cols1)
directed_hausdorff(mat0, mat1)

(1.4516636653256352, 10298, 14907)

In [6]:
def hausdorff_distance(P,Q):
    start_time=time.time()
    #inputs P and Q are arrays of vert coordinates

    dist = np.zeros((P.shape[0], 1))

    for p in range(P.shape[0]):

        # Calculate the minimum distance from points in P to Q

        minP = np.min(np.sum((P[p, :] - Q)**2, axis=1))

        dist[p, 0] = minP



    hd = np.sqrt(dist)
    end_time = time.time()
    return hd


In [58]:
def jaccard_similarity1(P_points, Q_points, resolution_xx, resolution_yy, resolution_zz) -> np.float32:
    x_min = min(min(P_points[:, 0]), min(Q_points[:, 0])); x_max = max(max(P_points[:, 0]), max(Q_points[:, 0]))
    y_min = min(min(P_points[:, 1]), min(Q_points[:, 1])); y_max = max(max(P_points[:, 1]), max(Q_points[:, 1]))
    z_min = min(min(P_points[:, 2]), min(Q_points[:, 2])); z_max = max(max(P_points[:, 2]), max(Q_points[:, 2]))
    
    xx = np.arange(x_min+.25, x_max-.25, resolution_xx) 
    yy = np.arange(y_min+.25, y_max-.25, resolution_yy)
    zz = np.arange(z_min+.25, z_max-.25, resolution_zz)
    s=time.time()
    xx, yy, zz = np.meshgrid(xx,yy,zz)
    grid = np.array([xx.ravel(),yy.ravel(),zz.ravel()]).T
    e=time.time()
    print(s-e)
    
    print(f"Size of grid: {grid.shape}")
    bool_grid_p = np.zeros(len(grid), dtype=bool)
    int_grid_p = np.zeros(len(grid))
    bool_grid_q = np.zeros(len(grid), dtype=bool)
    int_grid_q = np.zeros(len(grid))
    
    or_val = 0
    min_val = 0
    
    and_val = 0
    max_val = 0
    
    
    #P iteration
    for i in range(len(P_points)):
        grid_dist = np.sum((grid - P_points[i,:])**2, axis=1)
        bool_grid_p[np.argmin(grid_dist)] = True
        int_grid_p[np.argmin(grid_dist)] +=1
    return int_grid_p

In [ ]:
gcc -c -fpic c_sum.c
gcc -shared -o haus_40.dll c_sum.o